# Rubin alert archive in HATS

This notebook explores the Rubin alert archive and its transformation into HATS.

In [ ]:
%pip install lsdb hats --upgrade

In [ ]:
import lsdb
import pyarrow as pa
import pyarrow.compute as pc

from dask.distributed import Client
from hats_import import pipeline_with_client, ImportArguments
from hats_import.catalog.file_readers import ParquetPyarrowReader
from upath import UPath

### What we know

- There are currently ~13M alerts since Nov 2025 (according to [Fink](https://lsst.fink-portal.org/stats)):
    <img src="images/fink_stats.png" width="800"/>
- The alert packet format is defined in [sqr-114.lsst.io/v/DM-54197](https://sqr-114.lsst.io/v/DM-54197/index.html#alert-packet-format).
- The alerts are stored in compressed AVRO Confluent Wire Format (`.avro.gz`):
    ```bash
    Byte 0:     0x00          - magic byte, identifies Confluent Wire Format
    Bytes 1-4:  <uint32 BE>   - schema ID
    Bytes 5+:   <avro binary> - Avro-encoded record (no schema embedded)
    ```
- Each alert is stored in its individual file on disk. The files are named after the `alertId` (`diaSourceId`) of the alert it stores.

- An alert is encoded with a specific `schema`. There are currently two schemas - ID: `1000` and `1100`. The most recent schema, `1100` is backwards compatible with the previous one, since it only changes the nullability to some fields, which arrow already supports by default.

- The alerts are stored in `/sdf/data/rubin/shared/alert-archive/v2/alerts`.

- The schemas are stored in `/sdf/data/rubin/shared/alert-archive/v2/schemas`.

### Processing an alert file

Below is an example of reading with `fastavro` and converting to arrow with `avrotize`.

In [ ]:
apdb_path = UPath("/sdf/data/rubin/shared/alert-archive/v2")

In [ ]:
import fastavro
import json

from avrotize.avrotoparquet import AvroToParquetConverter

def load_input_schemas():
    """Load all AVRO schemas from files into a cache"""
    avro_schemas, arrow_schemas = {}, {}
    for path in (apdb_path / "schemas").glob("*.json"):
        schema_id = int(path.stem)
        with open(path) as f:
            schema = fastavro.parse_schema(json.load(f))
        avro_schemas[schema_id] = schema
        arrow_schemas[schema_id] = avro2arrow(schema)
    print(f"Found {len(avro_schemas)} schemas: {avro_schemas.keys()}")
    return avro_schemas, arrow_schemas

def avro2arrow(avro_schema):
    """Converts avro schemas into arrow schemas"""
    converter = AvroToParquetConverter()
    converter.cache_named_types(avro_schema)
    return pa.schema([
        pa.field(f["name"], converter.convert_avro_type_to_parquet_type(f["type"]))
        for f in avro_schema["fields"]
    ])

avro_schemas, arrow_schemas = load_input_schemas()

In [ ]:
from deepdiff import DeepDiff
print(DeepDiff(avro_schemas[1000], avro_schemas[1100], verbose_level=2).pretty())

In [ ]:
import io
import gzip
import struct

def read_alert(path):
    """Read single alert file"""
    with gzip.open(path, "rb") as f:
        data = f.read()
    schema_id = struct.unpack(">I", data[1:5])[0]
    chunk = fastavro.schemaless_reader(io.BytesIO(data[5:]), avro_schemas[schema_id])
    return pa.Table.from_pylist([chunk], schema=arrow_schemas[schema_id])

read_alert("/sdf/data/rubin/shared/alert-archive/v2/alerts/17006/170063701495775239.avro.gz").schema

We can create a custom file reader for HATS import to ingest these files. However... 

At one alert per file we are **I/O bound**. Reading millions of files one at a time is already slow, and we would duplicate the expensive work of transforming AVRO into Arrow in both the mapping and splitting stages. 

Neven ran a separate script to create a parquet dump of aggregated alerts.

In [ ]:
# Path to the alert archive dump on USDF
input_path = UPath("/sdf/data/rubin/user/ncaplar/alert-parquet")
# The output directory for the HATS collections
output_path = UPath("/sdf/data/rubin/shared/lsdb_commissioning")

We still need a custom alert reader to:
- Create base coordinate columns based on the alert DIA source ra/dec.
- Convert dict columns into structs of list so they are inferred as nested.
- Drop the cutouts that we do not need.

In [ ]:
class AlertReader(ParquetPyarrowReader):
    """Reads alert parquet files"""

    def read(self, input_file, **kwargs):
        for table in super().read(input_file):
            table = set_coordinates(table)
            table = convert_to_nested(table)
            columns_to_drop = [f for f in table.column_names if f.startswith("cutout")]
            yield table.drop_columns(columns_to_drop)

def set_coordinates(table):
    dia_source = table.column("diaSource")
    for col_name in ["dec","ra"]:
        coords = pc.struct_field(dia_source, col_name)
        table = table.add_column(0, col_name, coords)
    return table

def convert_to_nested(table):
    for i, field in enumerate(table.schema):
        if pa.types.is_struct(field.type):
            new_array = scalar_struct_to_struct_of_list(table.column(i))
        elif pa.types.is_list(field.type) and pa.types.is_struct(field.type.value_type):
            new_array = list_of_struct_to_struct_of_list(table.column(i))
        else:
            continue
        table = table.set_column(i, pa.field(field.name, new_array.type), new_array)
    return table

def list_of_struct_to_struct_of_list(array):
    if isinstance(array, pa.ChunkedArray):
        array = array.combine_chunks()
    offsets = array.offsets
    struct = array.values
    arrays = [pa.ListArray.from_arrays(offsets, struct.field(f.name)) for f in struct.type]
    return pa.StructArray.from_arrays(arrays, names=[f.name for f in struct.type])

def scalar_struct_to_struct_of_list(array):
    if isinstance(array, pa.ChunkedArray):
        array = array.combine_chunks()
    offsets = pa.array(range(len(array) + 1), type=pa.int32())
    arrays = [pa.ListArray.from_arrays(offsets, array.field(f.name)) for f in array.type]
    return pa.StructArray.from_arrays(arrays, names=[f.name for f in array.type])

Let's try the new parquet reader:

In [ ]:
import nested_pandas as npd
table = next(AlertReader().read(input_path / "170587__00100000-00138048.parquet"))
npd.from_pyarrow(table).head(1)

In [ ]:
args = ImportArguments(
    output_path=output_path,
    output_artifact_name="rubin_alert_archive",
    input_file_list=list(input_path.rglob("*.parquet")),
    file_reader=AlertReader(),
    pixel_threshold=200_000,
    resume=True,
)
with Client(
    n_workers=8,
    memory_limit="40GiB",
    threads_per_worker=1,
    local_directory=output_path / "dask_spill",
) as client:
    pipeline_with_client(args, client)

Let's have a quick look at the results:

In [ ]:
alerts = lsdb.open_catalog(output_path / "rubin_alert_archive")
alerts

In [ ]:
alerts.plot_coverage()
print(f"{len(alerts):,} alerts in the catalog")

The next step is to keep the catalog up-to-date with nightly increments.